# FlashEats — Class 8
## Building a Simple, Dependable Data Pipeline

### The story

Yesterday, our FlashEats workflow model worked.

We could calculate:

- Late Delivery Rate
- Order-to-Pickup Time
- Transit Time
- Customer interaction signals
- Intervention metrics

Today the client asks:

> **“Great. Can this run every morning without you?”**

That changes the engineering problem.

A successful notebook run is not the same thing as a dependable pipeline.

Today's goal:

```text
ONE COMMAND
    ↓
EXTRACT
    ↓
VALIDATE
    ↓
CLEAN
    ↓
TRANSFORM
    ↓
SAVE
    ↓
LOG
```

and if something is unsafe:

```text
FAIL CLEARLY
+
DO NOT PUBLISH BAD OUTPUT
```

In [1]:
# Dependencies are installed from Class8_Project/requirements.txt in section 8.

In [2]:
# SETUP — local first, Colab upload only if necessary.
# No matplotlib is required for this walkthrough.

import json
import os
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import pandas as pd


def find_pack_root():
    cwd = Path.cwd()

    # Fast path: notebook is already inside the pack.
    if (cwd / "database" / "flasheats.db").exists():
        return cwd

    # Common folder names.
    for name in [
        "FlashEats_Classroom_Pack_V2",
        "FlashEats_Classroom_Pack",
    ]:
        candidate = cwd / name
        if (candidate / "database" / "flasheats.db").exists():
            return candidate

    # Search locally, then Colab.
    for root in [cwd, cwd.parent, Path("/content")]:
        if not root.exists():
            continue

        for candidate in root.rglob("*"):
            if (
                candidate.is_dir()
                and (candidate / "database" / "flasheats.db").exists()
            ):
                return candidate

    return None


BASE = find_pack_root()

if BASE is None:
    try:
        from google.colab import files

        print("Upload FlashEats_Class8_Classroom_Pack.zip")
        uploaded = files.upload()
        zip_name = next(
            name for name in uploaded
            if name.endswith(".zip")
        )

        extract_dir = Path("/content/flasheats_class8")
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall(extract_dir)

        BASE = find_pack_root()

    except Exception as exc:
        print("Automatic setup failed:", exc)

if BASE is None:
    raise FileNotFoundError(
        "Could not locate a FlashEats pack containing database/flasheats.db"
    )

PROJECT = BASE / "Class8_Project"

print("Pack:", BASE)
print("Project:", PROJECT)
print("Database exists:", (BASE / "database" / "flasheats.db").exists())
print("run_pipeline.py exists:", (PROJECT / "run_pipeline.py").exists())

Pack: C:\Users\Ishan Ganguly\Downloads\FlashEats_Classwork_Classes_5-8\FlashEats_Classwork_Classes_5-8\Class8
Project: C:\Users\Ishan Ganguly\Downloads\FlashEats_Classwork_Classes_5-8\FlashEats_Classwork_Classes_5-8\Class8\Class8_Project
Database exists: True
run_pipeline.py exists: True


# 1. Working once ≠ dependable

Imagine yesterday's notebook contains:

```python
orders = load_orders()
dispatch = fetch_dispatch()
journey = build_journey(orders, dispatch)
journey.to_csv("order_journey.csv")
```

It works today.

## What assumptions are hidden?


1. What if the API returns only page 1?
2. What if the API returns HTTP 500?
3. What if `promised_eta` disappears?
4. What if the source is stale?
5. What if the job is run twice?
6. What if it crashes halfway through?

### Deduction

> **Dependability means making important assumptions explicit and testable.**

### Answers — hidden assumptions, and how the pipeline handles each

| # | Hidden assumption | What goes wrong in the notebook version | How `run_pipeline.py` handles it |
|---|---|---|---|
| 1 | The API returns everything in one response | Only page 1 (200 of 1,600 records) is loaded, and every metric is silently computed on 12.5% of orders | Loops while `has_more` is true, then checks the record count against `total_records`. A mismatch raises `Dispatch retrieval incomplete` |
| 2 | The API always answers 200 | One HTTP 500 crashes the run, or worse, a half-built dataframe is used | Retries 429/5xx up to `MAX_RETRIES=3` with backoff. Page 3 (500) and page 5 (429) both recovered on attempt 2 |
| 3 | The schema never changes | A later line crashes with a `KeyError`, or a NULL column gives a misleading late rate | The required-column gate stops the run (exit 2) with `missing required columns: ['promised_eta']` |
| 4 | The source is current | Old data is published as if it were today's | Freshness gate: newest `created_at` must be within `MAX_DATA_AGE_DAYS=60` of the run date |
| 5 | It only runs once | Appending doubles the rows on every rerun | Writes to a `run_date=` partition and replaces it. The rerun kept 1,600 rows with identical hashes |
| 6 | It never crashes midway | A partial file is left behind and read as if complete | Nothing is written to `processed/` until validation passes, and `order_journey.csv` uses a temp-file + `os.replace`. **Gap:** `metrics.json` and `validation_report.json` are written in place, so a crash between the three writes could leave a mixed partition |


# 2. Turn the notebook into stages

We will use five explicit stages:

```text
EXTRACT → VALIDATE → CLEAN → TRANSFORM → SAVE
```

Why stages?

Because when something fails, we want to know **what responsibility failed**.

Open the actual project structure:

In [3]:
for path in sorted(PROJECT.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT))

config\.env.example
data\processed\run_date=2026-09-22\metrics.json
data\processed\run_date=2026-09-22\order_journey.csv
data\processed\run_date=2026-09-22\validation_report.json
data\raw\dispatch\run_date=2026-09-22\dispatch_page_001.json
data\raw\dispatch\run_date=2026-09-22\dispatch_page_002.json
data\raw\dispatch\run_date=2026-09-22\dispatch_page_003.json
data\raw\dispatch\run_date=2026-09-22\dispatch_page_004.json
data\raw\dispatch\run_date=2026-09-22\dispatch_page_005.json
data\raw\dispatch\run_date=2026-09-22\dispatch_page_006.json
data\raw\dispatch\run_date=2026-09-22\dispatch_page_007.json
data\raw\dispatch\run_date=2026-09-22\dispatch_page_008.json
GATE2_DATA_READINESS_TEMPLATE.md
logs\pipeline_2026-09-22.log
logs\pipeline_2026-09-23.log
pipeline\__init__.py
pipeline\__pycache__\__init__.cpython-312.pyc
pipeline\__pycache__\clean.cpython-312.pyc
pipeline\__pycache__\config.cpython-312.pyc
pipeline\__pycache__\extract.cpython-312.pyc
pipeline\__pycache__\logging_utils.cpython-

### Discussion

- Why should validation happen before transformation?
- Why preserve raw API responses?
- Why is `save()` a separate stage?
- Would you put every operation in one giant function?

### Deduction

> Functions are not the goal. **Isolation of responsibility** is the goal.

### Answers

**Why should validation happen before transformation?** Transformations hide the evidence. Once `promised_eta` is turned into `delay_min`, or duplicates are dropped, you can't see that the source was broken. The result is a crash deep in `transform.py` or, worse, a plausible-looking wrong number. Validating raw inputs first means a failure names the real cause (`orders: missing required columns`) and nothing downstream runs on bad data.

**Why preserve raw API responses?** The API is an external system we can't replay. It may change, rate-limit us or go down. Saving each page under `data/raw/dispatch/run_date=…/` before processing lets us audit exactly what we received, re-run the transforms without calling the API again, and settle disputes like "the API never sent that order".

**Why is `save()` a separate stage?** It's the only stage with side effects visible to consumers. Keeping it separate makes the rule "publish only after validation succeeds" easy to enforce and check: `save_outputs` is the last call in the `try` block. It also keeps the idempotency logic (partition path, atomic replace) in one place.

**Would you put every operation in one giant function?** No. When a 400-line function fails, the traceback tells you a line number, not *which responsibility* failed. Separate stages can be tested on their own (`validate_freshness` with a fake dataframe), changed independently (adding a check doesn't touch transformation code), and logged per stage. They also make the failure types explicit: a `ValidationError` exits 2, anything unexpected exits 1.


# 3. Configuration should not be buried in code

Bad:

```python
API_URL = "http://localhost:8000"
MAX_RETRIES = 3
OUTPUT = "/Users/me/Desktop/output.csv"
```

Better:

```text
Code = behavior
Environment/config = where and how it runs
```

Let's inspect the example configuration.

In [4]:
print((PROJECT / "config" / ".env.example").read_text(encoding="utf-8"))

# FlashEats Class 8 pipeline configuration

DISPATCH_API_URL=http://127.0.0.1:8000
MAX_RETRIES=3
RETRY_BASE_SECONDS=1
MAX_DATA_AGE_DAYS=60
PAGE_SIZE=200
START_MOCK_API=true
LOG_LEVEL=INFO



### Discussion

1. Which values could differ between local, staging and production?
2. Which values should never be committed if they are secrets?
3. Why is `MAX_RETRIES` configuration but retry logic code?

### Deduction

> A pipeline is easier to move between environments when deployment-specific values are outside core business logic.

### Answers

**1. Values that differ between local, staging and production:** `DISPATCH_API_URL` (localhost mock vs the real Dispatch service), `START_MOCK_API` (true locally, *false* anywhere real), `LOG_LEVEL` (DEBUG locally, INFO in production), `PAGE_SIZE` (tuned to the real API's limits), `MAX_RETRIES` / `RETRY_BASE_SECONDS` (a real API may need longer backoff) and `MAX_DATA_AGE_DAYS` (60 suits a classroom snapshot, but a daily production job would want 1–2 days). The pack and output locations would also differ.

**2. Values that must never be committed:** API keys or tokens for the Dispatch service, database passwords or connection strings with credentials, and any cloud storage keys. None exist in this classroom project, which is why `.env.example` can be committed. In a real deployment they would be injected by the environment or a secrets manager, and `.env` would be git-ignored.

**3. Why `MAX_RETRIES` is configuration but retry logic is code:** the *policy* (how many attempts, how long to wait) is an operational choice that differs by environment and can be tuned without a release. The *mechanism* (which status codes are retryable, respecting `Retry-After`, exponential backoff, raising a clear error at the end) is behaviour that must be reviewed and tested. Changing *what* counts as retryable is a code change. Changing *how many times* is not.


# 4. Validation becomes a gate

Class 6 taught us how to discover data-quality rules.

Class 8 changes the question:

> **How do we make those rules execute every run?**

Our pipeline checks:

- required columns,
- duplicate order IDs,
- critical missing completion timestamps,
- freshness,
- required Dispatch fields.

Let's inspect the validation module.

In [5]:
validation_code = (PROJECT / "pipeline" / "validate.py").read_text(encoding="utf-8")

print(validation_code[:5000])

from dataclasses import dataclass
from datetime import date
import pandas as pd


class ValidationError(ValueError):
    pass


@dataclass
class CheckResult:
    check: str
    status: str
    detail: str


ORDER_REQUIRED_COLUMNS = {
    "order_id",
    "customer_id",
    "restaurant_id",
    "created_at",
    "promised_eta",
    "pickup_at",
    "actual_delivery_at",
    "final_status",
}


def validate_required_columns(df, required_columns, dataset_name):
    missing = sorted(set(required_columns) - set(df.columns))

    if missing:
        raise ValidationError(
            f"{dataset_name}: missing required columns: {missing}"
        )

    return CheckResult(
        check=f"{dataset_name}.required_columns",
        status="PASS",
        detail=f"{len(required_columns)} required columns present",
    )


def validate_order_uniqueness(orders):
    duplicate_rows = int(orders.duplicated("order_id", keep=False).sum())

    # This source is intentionally known to contain a few dupli

### Story

Suppose `promised_eta` disappears tomorrow.

A weak pipeline continues until some later line crashes — or worse, produces a misleading output.

A dependable pipeline should say:

```text
VALIDATION FAILED
orders: missing required columns: ['promised_eta']

No processed output written.
```

### Deduction

> **Validation protects downstream consumers from silently bad output.**

# 5. Retries: retry the right failures

The FlashEats Dispatch API deliberately behaves like a real external dependency:

- page 3 fails once with HTTP 500,
- page 5 rate-limits once with HTTP 429,
- pagination must be completed.

A retry is appropriate for a **transient** failure.

A retry is usually not useful for:

- missing required column,
- invalid credentials,
- incompatible schema,
- a business-rule failure.

Let's inspect the retry logic.

In [6]:
extract_code = (PROJECT / "pipeline" / "extract.py").read_text(encoding="utf-8")

start = extract_code.find("def _request_page")
end = extract_code.find("def extract_dispatch_api")

print(extract_code[start:end])

def _request_page(session, url, page, page_size, max_retries, base_seconds, logger):
    for attempt in range(1, max_retries + 1):
        try:
            response = session.get(
                f"{url}/dispatch/orders",
                params={"page": page, "page_size": page_size},
                timeout=5,
            )

            if response.status_code == 200:
                return response.json()

            if response.status_code in {429, 500, 502, 503, 504}:
                retry_after = response.headers.get("Retry-After")
                if response.status_code == 429:
                    try:
                        body = response.json()
                        retry_after = body.get(
                            "retry_after_seconds",
                            retry_after,
                        )
                    except Exception:
                        pass

                wait_seconds = (
                    float(retry_after)
                    if retry_af

### Discussion

1. Why limit the number of retries?
2. Why wait longer between repeated attempts?
3. Why should HTTP 429 and HTTP 500 be treated differently from a schema failure?
4. Could retrying a write operation be dangerous?

### Deduction

> **Retry transient failures; fail fast on conditions that require human or code changes.**

### Answers

**1. Why limit the number of retries?** An unlimited retry turns an outage into a job that never finishes. It holds resources, delays the alert, and hammers a service that is already struggling. A bounded limit (`MAX_RETRIES=3`) gives transient failures a fair chance, then fails with a clear message. In the unreachable-API test, the pipeline stopped after 3 attempts with `Dispatch API page 1 failed after 3 attempts` and published nothing.

**2. Why wait longer between repeated attempts?** If the first retry fails, the problem probably isn't a one-off blip. Exponential backoff (`base × 2^(attempt−1)`: 1s, 2s, 4s…) gives the dependency time to recover, and it avoids many clients retrying in lockstep against an overloaded service. For HTTP 429 the server tells us how long to wait, so the code uses `retry_after_seconds` / `Retry-After` instead of guessing.

**3. Why treat HTTP 429 and 500 differently from a schema failure?** 429 ("slow down") and 500 ("temporary server error") are *transient*, so the same request can succeed a moment later, as pages 3 and 5 did. A missing `promised_eta` is *deterministic*: retrying gives the same broken data every time, only later. That needs a human or a code change, so the pipeline raises `ValidationError` immediately (exit 2) instead of retrying.

**4. Could retrying a write operation be dangerous?** Yes. If a `POST /refund` times out *after* the server processed it, retrying can issue a second refund, send a duplicate message or reassign a driver twice. That's safe only if the operation is idempotent, for example through an idempotency key the server uses to de-duplicate, or a "check then write" step. This pipeline only retries `GET` reads, which have no side effects, so retrying them is safe.


# 6. Idempotency: what happens if we run twice?

A job that appends the same 1,600 orders every time is repeatable — but not dependable.

For this class:

```text
same logical run_date
        ↓
same output partition
        ↓
atomically replace output
```

Let's inspect the save stage.

In [7]:
save_code = (PROJECT / "pipeline" / "save.py").read_text(encoding="utf-8")
print(save_code)

from pathlib import Path
import json
import os
import tempfile


def atomic_write_csv(df, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)

    with tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".csv",
        delete=False,
        dir=destination.parent,
    ) as tmp:
        temp_path = Path(tmp.name)

    try:
        df.to_csv(temp_path, index=False)
        os.replace(temp_path, destination)
    finally:
        if temp_path.exists():
            temp_path.unlink(missing_ok=True)


def save_outputs(
    journey,
    metrics,
    validation_results,
    project_root: Path,
    run_date: str,
    logger,
):
    partition = (
        project_root
        / "data"
        / "processed"
        / f"run_date={run_date}"
    )

    partition.mkdir(parents=True, exist_ok=True)

    journey_path = partition / "order_journey.csv"
    metrics_path = partition / "metrics.json"
    validation_path = partition / "validation_report.json"

    # Idemp

### Discussion

1. Why partition outputs by logical run date?
2. Why replace instead of append for this pipeline?
3. When would append actually be correct?
4. Why write a temporary file before replacing the final file?

### Deduction

> **A rerun should not create unintended duplicate effects.**

### Answers

**1. Why partition outputs by logical run date?** The partition name comes from *which day's data this is* (`--run-date 2026-09-22`), not from when the job happened to run. A rerun, backfill or late retry of the 22nd therefore lands in the same place, rather than creating a new file stamped with today's time. It also makes each run easy to find, compare and roll back independently.

**2. Why replace instead of append for this pipeline?** Each run rebuilds the *complete* order-journey snapshot for its run date (all 1,600 orders). Appending would add a second copy of every order on each rerun, so one row per order would become two, and every downstream count and rate would be wrong. Replacing makes the rerun safe: the second run above kept 1,600 rows and identical hashes.

**3. When would append actually be correct?** When each run processes *new, non-overlapping* records, for example an event log or an incremental load of today's new orders. Even then, the rerun must be protected by a unique key with an upsert/merge, or by only appending records newer than a stored watermark. Otherwise a retry still duplicates them.

**4. Why write a temporary file before replacing the final file?** Writing straight to the final path means a crash, full disk or kill signal can leave a half-written `order_journey.csv` that readers treat as complete. Writing to a temp file in the *same directory* and then calling `os.replace` swaps it in one atomic step, so readers see either the old complete file or the new complete file, never a partial one. **Gap in this project:** only the CSV does this. `metrics.json` and `validation_report.json` are written in place, so the partition as a whole isn't atomic.


# 7. Logging should help the next engineer

Bad log:

```text
ERROR
```

Useful log:

```text
Dispatch API failure
page=3
status=500
attempt=2/3
```

A log should help answer:

- which stage?
- which source?
- how many records?
- which attempt?
- what failed?
- did output get published?

The pipeline writes both console and file logs.

In [8]:
print((PROJECT / "pipeline" / "logging_utils.py").read_text(encoding="utf-8"))

import logging
from pathlib import Path


def build_logger(log_path: Path, level: str = "INFO") -> logging.Logger:
    log_path.parent.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger("flasheats_pipeline")
    logger.setLevel(getattr(logging, level, logging.INFO))
    logger.handlers.clear()

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    console = logging.StreamHandler()
    console.setFormatter(formatter)

    file_handler = logging.FileHandler(log_path)
    file_handler.setFormatter(formatter)

    logger.addHandler(console)
    logger.addHandler(file_handler)

    return logger



### Deduction

> **A dependable pipeline is diagnosable, not merely executable.**

# 8. Run the complete flow with one command

This is the handoff moment.

From `Class8_Project`:

```bash
python run_pipeline.py --run-date 2026-09-22
```

The command should:

1. start the mock API if needed,
2. extract local sources,
3. retrieve every Dispatch API page,
4. retry transient failures,
5. preserve raw pages,
6. validate inputs,
7. clean the agreed business grain,
8. build the Class 7 order journey,
9. save metrics + validation report,
10. write logs.

Run it from this notebook:

In [9]:
# Install only the small dependencies used by the project.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")],
    check=True,
)

result = subprocess.run(
    [
        sys.executable,
        str(PROJECT / "run_pipeline.py"),
        "--run-date",
        "2026-09-22",
    ],
    cwd=PROJECT,
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print("\nSTDOUT:\n", result.stdout[-5000:])
print("\nSTDERR:\n", result.stderr[-3000:])

RETURN CODE: 0

STDOUT:
 
PIPELINE SUCCESS
{
  "orders_in_journey": 1600,
  "measurable_deliveries": 1495,
  "late_delivery_rate_pct": 56.39,
  "median_order_to_pickup_min": 26.07,
  "median_transit_min": 47.23,
  "orders_with_intervention": 430
}

Outputs:
  journey: C:\Users\Ishan Ganguly\Downloads\FlashEats_Classwork_Classes_5-8\FlashEats_Classwork_Classes_5-8\Class8\Class8_Project\data\processed\run_date=2026-09-22\order_journey.csv
  metrics: C:\Users\Ishan Ganguly\Downloads\FlashEats_Classwork_Classes_5-8\FlashEats_Classwork_Classes_5-8\Class8\Class8_Project\data\processed\run_date=2026-09-22\metrics.json
  validation: C:\Users\Ishan Ganguly\Downloads\FlashEats_Classwork_Classes_5-8\FlashEats_Classwork_Classes_5-8\Class8\Class8_Project\data\processed\run_date=2026-09-22\validation_report.json


STDERR:
 2026-09-27 16:49:56,218 | INFO | Pipeline started | run_date=2026-09-22 chaos=none
2026-09-27 16:49:57,728 | INFO | Starting local mock dispatch API
2026-09-27 16:49:58,262 | INFO

# 9. Prove idempotency

Run exactly the same logical job again.

What should happen?

- output row count should not double,
- the same partition should be replaced,
- the final business dataset should remain one row per order.

In [10]:
output_path = (
    PROJECT
    / "data"
    / "processed"
    / "run_date=2026-09-22"
    / "order_journey.csv"
)

first = pd.read_csv(output_path)
first_rows = len(first)

second_run = subprocess.run(
    [
        sys.executable,
        str(PROJECT / "run_pipeline.py"),
        "--run-date",
        "2026-09-22",
    ],
    cwd=PROJECT,
    text=True,
    capture_output=True,
)

second = pd.read_csv(output_path)
second_rows = len(second)

print("First run rows:", first_rows)
print("Second run rows:", second_rows)
print("Rows doubled?", second_rows == first_rows * 2)
print("Same logical row count?", first_rows == second_rows)

First run rows: 1600
Second run rows: 1600
Rows doubled? False
Same logical row count? True


### Deduction

> **“Can rerun” and “safe to rerun” are different properties.**

# 10. Controlled failure: break the schema

A dependable pipeline should not just succeed nicely.

It should fail nicely.

Run:

In [11]:
failure = subprocess.run(
    [
        sys.executable,
        str(PROJECT / "run_pipeline.py"),
        "--run-date",
        "2026-09-22",
        "--chaos",
        "missing_column",
    ],
    cwd=PROJECT,
    text=True,
    capture_output=True,
)

print("RETURN CODE:", failure.returncode)
print(failure.stdout[-3000:])

RETURN CODE: 2

PIPELINE FAILED: orders: missing required columns: ['promised_eta']



### Discuss

1. Did the pipeline explain what was wrong?
2. Did it keep retrying a non-retryable problem?
3. Did it publish a new processed dataset?
4. What would an on-call engineer need next?

### Deduction

> **Dependability includes safe, understandable failure.**

### Answers (from the `missing_column` run above)

**1. Did the pipeline explain what was wrong?** Yes. It printed `PIPELINE FAILED: orders: missing required columns: ['promised_eta']`, which names the dataset, the check and the exact column. The log adds `Pipeline stopped at validation gate … no processed output written`. The distinct return code (2 for a validation failure, 1 for an unexpected error) lets a scheduler tell the two apart.

**2. Did it keep retrying a non-retryable problem?** No. The Dispatch pages were fetched normally (the 500 and 429 errors were retried because they're transient), but the schema failure raised `ValidationError` once and stopped the run immediately. Retrying would only have fetched the same broken schema again.

**3. Did it publish a new processed dataset?** No. The files in `data/processed/run_date=2026-09-22/` kept the same MD5 hashes before and after the failed run, so consumers still see the last good output. The raw Dispatch pages *were* saved, which is intended: raw evidence is preserved even when the run fails.

**4. What would an on-call engineer need next?**
- **What changed upstream:** which system dropped `promised_eta`, and whether it was renamed, removed or broken by a deploy. That means a source owner to contact.
- **The last good run:** which partition consumers are currently reading, and whether it's too old to leave in place.
- **An alert, not just a log line:** a scheduler that sends an alert on exit code 2 with the error message.
- **A runbook entry:** "schema validation failed → don't re-run blindly; confirm the schema change with the source owner, then update `ORDER_REQUIRED_COLUMNS` / the mapping, or wait for the upstream fix."


# 11. Gate 2 — Data readiness review

The final output of this module is not merely `pipeline.py`.

We need to be able to answer:

> **Is the data foundation ready for the next product/AI phase?**

Review the template:

In [12]:
print((PROJECT / "GATE2_DATA_READINESS_TEMPLATE.md").read_text(encoding="utf-8"))

# Gate 2 — Data Readiness Review

**Pipeline:** FlashEats order-journey pipeline · **Run date reviewed:** `2026-09-22` · **Command:** `python run_pipeline.py --run-date 2026-09-22`

**Data:** the FlashEats classroom pack from `github.com/manangupta12/flasheats-classroom-pack` (commit `0105c7a`). All evidence below was re-run against it on 2026-09-27.

## Pipeline run

- [x] Complete flow runs with one command. The command starts the mock Dispatch API, extracts, validates, cleans, transforms, saves and logs, then exits 0 with `PIPELINE SUCCESS`.
- [x] Same run can be safely repeated. A second run for the same date produced byte-identical `order_journey.csv`, `metrics.json` and `validation_report.json` (same MD5 hashes), and the row count stayed at 1,600.
- [x] Raw API responses are preserved. 8 pages are saved under `data/raw/dispatch/run_date=2026-09-22/dispatch_page_001–008.json` before any transformation.
- [x] Processed output is written only after validation. In the `missing_column

# The story to carry out of Class 8

We started with:

```text
“Here is a notebook that works.”
```

We ended with:

```text
ONE COMMAND
   ↓
KNOWN CONFIGURATION
   ↓
BOUNDED RETRIES
   ↓
VALIDATION GATES
   ↓
EXPLICIT TRANSFORMS
   ↓
IDEMPOTENT OUTPUT
   ↓
USEFUL LOGS
   ↓
READINESS DECISION
```

The core lesson:

> **A dependable pipeline is not one that never fails. It produces trustworthy output when things are healthy, and fails clearly and safely when they are not.**

# Knowledge check

## Poll 1
The Dispatch API returns HTTP 500 once. What should the pipeline usually do?

A. Delete the input  
B. Retry a bounded number of times  
C. Remove validation  
D. Retry forever


---

## Poll 2
`promised_eta` disappears from the source schema. What should happen?

A. Retry the API  
B. Fill the column with NULL and publish  
C. Fail the validation gate with a useful message  
D. Ignore it because yesterday's run worked


---

## Poll 3
A pipeline run for the same `run_date` is executed twice. Which behavior best demonstrates idempotency?

A. Double the rows  
B. Create two identical permanent copies  
C. Replace the same logical output without duplicate effects  
D. Refuse every rerun


---

## Poll 4
Which is the best log message?

A. `ERROR`  
B. `Something broke`  
C. `Dispatch page=5 status=429 attempt=2/3 wait=1s`  
D. `Try again`


---

## Poll 5
Which Class 6 concept changes meaningfully in Class 8?

A. We stop validating data  
B. Data-quality rules become executable gates on every run  
C. We replace business rules with infrastructure  
D. We remove source ownership

# Knowledge check — answers

| Poll | Answer | Why |
|---|---|---|
| 1. HTTP 500 once | **B. Retry a bounded number of times** | A 500 is transient. Retrying forever hides a real outage; the run above recovered page 3 on the second attempt |
| 2. `promised_eta` disappears | **C. Fail the validation gate with a useful message** | A retry can't fix a schema break, and filling it with NULL would publish a misleading late rate. The chaos run above stopped with `missing required columns: ['promised_eta']` and wrote no output |
| 3. Same `run_date` twice | **C. Replace the same logical output without duplicate effects** | The rerun above produced byte-identical files and the row count stayed at 1,600 |
| 4. Best log message | **C. `Dispatch page=5 status=429 attempt=2/3 wait=1s`** | It names the stage, source, page, status and attempt, which is everything an on-call engineer needs |
| 5. Class 6 concept that changes | **B. Data-quality rules become executable gates on every run** | Class 6 discovered the rules once; Class 8 runs them before every publish |
